# 02 · Camada Silver — Transformação (T) e SCD Tipo 2

A camada silver recebe os dados do bronze e aplica **limpeza, normalização e enriquecimento**, produzindo tabelas prontas para a camada gold.

**Transformações aplicadas:**
- Resolução de NULL com valores padrão (brand → 'Unknown', size/color → 'N/A').
- JOINs para desnormalizar descrições (colorname, stockgroupname).
- Trim em colunas de texto.
- Conversão de `isfinalized` (boolean → SMALLINT 0/1).
- **SCD Tipo 2** na `dim_stockitems_scd2`: historial completo de alterações com `valid_from`, `valid_to` e `is_current`.

## 1. Imports e ligações

In [1]:
import pandas as pd
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

# Ligação ao Data Warehouse
engine_dwh = create_engine(
    f"postgresql+psycopg2://{os.getenv('DWH_USER')}:{os.getenv('DWH_PASSWORD')}@"
    f"{os.getenv('DWH_HOST')}:{os.getenv('DWH_PORT')}/{os.getenv('DWH_DB')}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
SCD2_SENTINEL = date(9999, 12, 31)

print(f"✓ Ligação estabelecida. Início do processamento: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")

✓ Ligação estabelecida. Início do processamento: 2026-04-16 18:40:27 UTC


In [11]:
with engine_dwh.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS 
            silver.dim_stockitems_scd2, silver.stg_stockitems, silver.stg_customers, 
            silver.stg_geography, silver.stg_salespersons, silver.stg_invoices, 
            silver.stg_invoicelines, silver.stg_orders, silver.stg_orderlines, 
            silver.stg_customertransactions CASCADE;
    """))
print("✓ Tabelas antigas da camada Silver apagadas com sucesso!")

✓ Tabelas antigas da camada Silver apagadas com sucesso!


## 2. Preparação das tabelas Silver

In [12]:
DDL_SILVER = """
-- 1. Tabela de Histórico (SCD Tipo 2)
CREATE TABLE IF NOT EXISTS silver.dim_stockitems_scd2 (
    scd_key            SERIAL PRIMARY KEY,
    stockitemid        INT NOT NULL,
    stockitemname      VARCHAR(255) NOT NULL,
    brand              VARCHAR(100),
    colorname          VARCHAR(50),
    size               VARCHAR(50),
    leadtimedays       INT,
    quantityperouter   INT,
    stockgroupname     VARCHAR(100),
    valid_from         DATE NOT NULL,
    valid_to           DATE NOT NULL DEFAULT '9999-12-31',
    is_current         BOOLEAN NOT NULL DEFAULT TRUE,
    _source_snapshot   INT NOT NULL,
    _loaded_at         TIMESTAMP NOT NULL
);

-- 2. Tabelas de Staging (Preparação)
CREATE TABLE IF NOT EXISTS silver.stg_stockitems (
    stockitemid INT, stockitemname VARCHAR(255), brand VARCHAR(100),
    colorname VARCHAR(50), size VARCHAR(50), leadtimedays INT,
    quantityperouter INT, stockgroupname VARCHAR(100), _source_snapshot INT, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_customers (
    customerid INT, customername VARCHAR(255), buyinggroupname VARCHAR(100),
    customercategoryname VARCHAR(100), creditlimit NUMERIC(18,2),
    accountopeneddate DATE, paymentdays INT, deliverycityid INT, _source_snapshot INT, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_geography (
    cityid INT, cityname VARCHAR(100), stateprovincename VARCHAR(100),
    countryname VARCHAR(100), latestrecordedpopulation BIGINT, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_salespersons (
    salespersonid INT, fullname VARCHAR(255), preferredname VARCHAR(255),
    phonenumber VARCHAR(50), _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_invoices (
    invoiceid INT, customerid INT, billtocustomerid INT, orderid INT,
    deliverymethodid INT, contactpersonid INT, salespersonpersonid INT, 
    packedbypersonid INT, invoicedate DATE, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_invoicelines (
    invoicelineid INT, invoiceid INT, stockitemid INT, description VARCHAR(255),
    quantity INT, unitprice NUMERIC(18,2), taxrate NUMERIC(18,3),
    lineprofit NUMERIC(18,2), extendedprice NUMERIC(18,2), invoicedate DATE, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_orders (
    orderid INT, customerid INT, salespersonpersonid INT, pickedbypersonid INT,
    contactpersonid INT, backorderorderid INT, orderdate DATE, expecteddeliverydate DATE,
    isundersupplybackordered INT, comments TEXT, deliveryinstructions TEXT,
    internalcomments TEXT, pickingcompletedwhen TIMESTAMP, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_orderlines (
    orderlineid INT, orderid INT, stockitemid INT, description VARCHAR(255),
    packagetypeid INT, quantity INT, unitprice NUMERIC(18,2), taxrate NUMERIC(18,3),
    pickedquantity INT, orderdate DATE, _loaded_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS silver.stg_customertransactions (
    customertransactionid INT, customerid INT, transactiondate DATE,
    transactionamount NUMERIC(18,2), outstandingbalance NUMERIC(18,2),
    isfinalized INT, _loaded_at TIMESTAMP
);

-- 3. Limpeza de Staging
TRUNCATE TABLE 
    silver.stg_stockitems, silver.stg_customers, silver.stg_geography, 
    silver.stg_salespersons, silver.stg_invoices, silver.stg_invoicelines, 
    silver.stg_orders, silver.stg_orderlines, silver.stg_customertransactions;
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_SILVER))

print("✓ Tabelas Silver garantidas e Staging limpo.")

✓ Tabelas Silver garantidas e Staging limpo.


## 3. Funções auxiliares

In [13]:
def read_active_snapshot(table_name: str) -> pd.DataFrame:
    sql = f"SELECT * FROM bronze.{table_name} WHERE _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name}) AND _change_op != 'DELETE'"
    with engine_dwh.connect() as conn:
        return pd.read_sql(text(sql), conn)

def clean_str(s: pd.Series) -> pd.Series:
    return s.str.strip().replace("", None)

## 4.Stock Items + SCD Tipo 2 (Tratamento de Nulos)

In [14]:
SCD2_COLS = ["stockitemname", "brand", "colorname", "size", "stockgroupname", "leadtimedays", "quantityperouter"]

df_items  = read_active_snapshot("stockitems")
df_colors = read_active_snapshot("colors")

if not df_items.empty:
    snap_id = int(df_items["_snapshot_id"].max())

    df = df_items.merge(df_colors[["colorid", "colorname"]], on="colorid", how="left")
    
    # TRATAMENTO DO DATA PROFILING: Valores nulos -> "N/A"
    df["brand"] = df["brand"].fillna("N/A").pipe(clean_str)
    df["colorname"] = df["colorname"].fillna("N/A").pipe(clean_str)
    df["size"] = df["size"].fillna("N/A").pipe(clean_str)
    df["stockitemname"] = clean_str(df["stockitemname"])

    silver_items = df[["stockitemid", "stockitemname", "brand", "colorname", "size", "leadtimedays", "quantityperouter"]].copy()
    silver_items["stockgroupname"] = "N/A"
    silver_items["_loaded_at"], silver_items["_source_snapshot"] = run_at, snap_id
    
    # Inserir no Staging
    silver_items.to_sql("stg_stockitems", engine_dwh, schema="silver", if_exists="append", index=False)

    # Lógica SCD2
    today = run_at.date()
    yesterday = date.fromordinal(today.toordinal() - 1)

    with engine_dwh.connect() as conn:
        df_current = pd.read_sql(text("SELECT * FROM silver.dim_stockitems_scd2 WHERE is_current = TRUE"), conn)

    if df_current.empty:
        to_insert = silver_items.copy()
        to_insert["valid_from"], to_insert["valid_to"], to_insert["is_current"] = today, SCD2_SENTINEL, True
        to_insert.to_sql("dim_stockitems_scd2", engine_dwh, schema="silver", if_exists="append", index=False)
        print(f"✓ SCD2: Carga inicial ({len(to_insert)} items).")
    else:
        merged = silver_items.merge(df_current, on="stockitemid", how="left", suffixes=('_new', '_old'))
        new_mask = merged['is_current'].isna()
        
        changed_mask = pd.Series(False, index=merged.index)
        for col in SCD2_COLS:
             changed_mask |= (merged[f"{col}_new"].fillna("").astype(str) != merged[f"{col}_old"].fillna("").astype(str))
        changed_mask &= ~new_mask

        if changed_mask.any():
            keys_to_expire = merged.loc[changed_mask, "scd_key"].tolist()
            with engine_dwh.begin() as conn:
                conn.execute(text("UPDATE silver.dim_stockitems_scd2 SET valid_to = :vt, is_current = FALSE WHERE scd_key = ANY(:keys)"), 
                             {"vt": yesterday, "keys": keys_to_expire})
            
        to_insert_df = merged[new_mask | changed_mask].copy()
        if not to_insert_df.empty:
            final_insert = pd.DataFrame()
            for c in SCD2_COLS + ["stockitemid"]: final_insert[c] = to_insert_df[f"{c}_new"]
            final_insert["valid_from"], final_insert["valid_to"], final_insert["is_current"] = today, SCD2_SENTINEL, True
            final_insert["_source_snapshot"], final_insert["_loaded_at"] = snap_id, run_at
            final_insert.to_sql("dim_stockitems_scd2", engine_dwh, schema="silver", if_exists="append", index=False)

        print(f"✓ SCD2 Processado: {new_mask.sum()} novos, {changed_mask.sum()} atualizados.")

✓ SCD2: Carga inicial (227 items).


## 5. Clientes (Tratamento de Nulos)

In [15]:
df_cust = read_active_snapshot("customers")
df_bg   = read_active_snapshot("buyinggroups")
df_cc   = read_active_snapshot("customercategories")

if not df_cust.empty:
    snap_id = int(df_cust["_snapshot_id"].max())
    df = df_cust.merge(df_bg[["buyinggroupid", "buyinggroupname"]], on="buyinggroupid", how="left")
    df = df.merge(df_cc[["customercategoryid", "customercategoryname"]], on="customercategoryid", how="left")

    # TRATAMENTO DO DATA PROFILING
    df["buyinggroupname"] = df["buyinggroupname"].fillna("Sem Grupo").pipe(clean_str)
    df["creditlimit"] = df["creditlimit"].fillna(0.00)
    
    df["customername"] = clean_str(df["customername"])

    silver_cust = df[["customerid", "customername", "buyinggroupname", "customercategoryname", "creditlimit", "accountopeneddate", "paymentdays", "deliverycityid"]].copy()
    silver_cust["_loaded_at"], silver_cust["_source_snapshot"] = run_at, snap_id
    
    silver_cust.to_sql("stg_customers", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_customers processado ({len(silver_cust)} clientes).")

✓ stg_customers processado (663 clientes).


## 6. Geografia

In [16]:
df_cities = read_active_snapshot("cities")
df_states = read_active_snapshot("stateprovinces")
df_countries = read_active_snapshot("countries")

if not df_cities.empty:
    df_geo = df_cities.merge(df_states[["stateprovinceid", "stateprovincename", "countryid"]], on="stateprovinceid", how="left")
    df_geo = df_geo.merge(df_countries[["countryid", "ountryname"]], on="countryid", how="left")
    df_geo.rename(columns={"ountryname": "countryname"}, inplace=True)
    
    silver_geo = df_geo[["cityid", "cityname", "stateprovincename", "countryname", "latestrecordedpopulation"]].copy()
    silver_geo["_loaded_at"] = run_at
    
    silver_geo.to_sql("stg_geography", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_geography criado ({len(silver_geo)} registos).")

✓ stg_geography criado (37940 registos).


## 7. Vendedores

In [17]:
df_people = read_active_snapshot("people")

if not df_people.empty:
    df_sales = df_people[df_people["issalesperson"] == 1].copy()
    silver_sales = df_sales[["personid", "fullname", "preferredname", "phonenumber"]].copy()
    silver_sales.rename(columns={"personid": "salespersonid"}, inplace=True)
    silver_sales["_loaded_at"] = run_at
    
    silver_sales.to_sql("stg_salespersons", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_salespersons preparado ({len(silver_sales)} vendedores).")

✓ stg_salespersons preparado (10 vendedores).


## 8. Faturas e Encomendas

In [ ]:
with engine_dwh.connect() as conn:
    df_inv = pd.read_sql(text("SELECT * FROM bronze.invoices"), conn)
    df_inv_lines = pd.read_sql(text("SELECT * FROM bronze.invoicelines"), conn)
    df_ord = pd.read_sql(text("SELECT * FROM bronze.orders"), conn)
    df_ord_lines = pd.read_sql(text("SELECT * FROM bronze.orderlines"), conn)

if not df_inv.empty:
    # Selecionar apenas as colunas que criámos no Staging (deitando fora o lixo do Bronze)
    cols_inv = ["invoiceid", "customerid", "billtocustomerid", "orderid", "deliverymethodid", "contactpersonid", "salespersonpersonid", "packedbypersonid", "invoicedate"]
    silver_inv = df_inv[cols_inv].copy()
    silver_inv["_loaded_at"] = run_at
    silver_inv.to_sql("stg_invoices", engine_dwh, schema="silver", if_exists="append", index=False)
    
    cols_inv_lines = ["invoicelineid", "invoiceid", "stockitemid", "description", "quantity", "unitprice", "taxrate", "lineprofit", "extendedprice", "invoicedate"]
    silver_inv_lines = df_inv_lines[cols_inv_lines].copy()
    silver_inv_lines["_loaded_at"] = run_at
    silver_inv_lines.to_sql("stg_invoicelines", engine_dwh, schema="silver", if_exists="append", index=False)
    
    print(f"✓ Staging de Faturas concluído: {len(silver_inv)} faturas e {len(silver_inv_lines)} linhas.")

if not df_ord.empty:
    cols_ord = ["orderid", "customerid", "salespersonpersonid", "pickedbypersonid", "contactpersonid", "backorderorderid", "orderdate", "expecteddeliverydate", "isundersupplybackordered", "comments", "deliveryinstructions", "internalcomments", "pickingcompletedwhen"]
    silver_ord = df_ord[cols_ord].copy()
    silver_ord["_loaded_at"] = run_at
    silver_ord.to_sql("stg_orders", engine_dwh, schema="silver", if_exists="append", index=False)
    
    cols_ord_lines = ["orderlineid", "orderid", "stockitemid", "description", "packagetypeid", "quantity", "unitprice", "taxrate", "pickedquantity", "orderdate"]
    silver_ord_lines = df_ord_lines[cols_ord_lines].copy()
    silver_ord_lines["_loaded_at"] = run_at
    silver_ord_lines.to_sql("stg_orderlines", engine_dwh, schema="silver", if_exists="append", index=False)
    
    print(f"✓ Staging de Encomendas concluído: {len(silver_ord)} encomendas e {len(silver_ord_lines)} linhas.")

## 9. Transações de Clientes

In [ ]:
with engine_dwh.connect() as conn:
    df_ct = pd.read_sql(text("SELECT * FROM bronze.customertransactions"), conn)

if not df_ct.empty:
    df_ct["isfinalized"] = df_ct["isfinalized"].fillna(0).astype(int) 
    silver_ct = df_ct[["customertransactionid", "customerid", "transactiondate", "transactionamount", "outstandingbalance", "isfinalized"]].copy()
    silver_ct["_loaded_at"] = run_at
    
    silver_ct.to_sql("stg_customertransactions", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_customertransactions processado ({len(silver_ct)} registos).")